In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# **(1) Load & Clean NASA FIRMS Fire Detection Data**

### Load the split FIRMS CSV files and combine into single dataframe

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

files = [
    '/content/drive/MyDrive/MDP_Capstone_Project/Data/fire_archive_SV-C2_2014-2015.csv',
    '/content/drive/MyDrive/MDP_Capstone_Project/Data/fire_archive_SV-C2_2016-2018.csv',
    '/content/drive/MyDrive/MDP_Capstone_Project/Data/fire_archive_SV-C2_2019-2022.csv',
    '/content/drive/MyDrive/MDP_Capstone_Project/Data/fire_archive_SV-C2_2023.csv',
    '/content/drive/MyDrive/MDP_Capstone_Project/Data/fire_archive_SV-C2_2024.csv']

fire_df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

Mounted at /content/drive


### Clean the merged NASA fire dataset

In [ ]:
# Check for missing values
print("Missing Fire Values:", fire_df.isnull().sum())

# Create index column
fire_df = fire_df.reset_index(drop=True)
fire_df['fire_id'] = range(1, len(fire_df) + 1)

# Ensure acq_date is datetime
fire_df['acq_date'] = pd.to_datetime(fire_df['acq_date'])

# Create columns for year, month, and day
fire_df['year'] = fire_df['acq_date'].dt.year
fire_df['month'] = fire_df['acq_date'].dt.month
fire_df['day_of_year'] = fire_df['acq_date'].dt.dayofyear
fire_df['hour'] = fire_df['acq_time'].astype(str).str.zfill(4).str[:2].astype(int)

# Download original combined df as CSV
fire_df.to_csv('/content/drive/MyDrive/MDP_Capstone_Project/Data/original_fire_df.csv', index=False)

# Show sample table of the original fire data
fire_df

Missing Fire Values: latitude      0
longitude     0
brightness    0
scan          0
track         0
acq_date      0
acq_time      0
confidence    0
bright_t31    0
frp           0
daynight      0
type          0
dtype: int64


,latitude,longitude,brightness,scan,track,acq_date,acq_time,confidence,bright_t31,frp,daynight,type,fire_id,year,month,day_of_year,hour
0,57.03862,-121.94064,320.09,0.75,0.77,2014-01-01,849,n,248.54,4.45,N,0,1,2014,1,1,8
1,56.82743,-121.11518,335.42,0.70,0.75,2014-01-01,849,n,245.92,6.46,N,0,2,2014,1,1,8
2,56.82357,-121.11824,328.37,0.70,0.75,2014-01-01,849,n,244.90,6.79,N,0,3,2014,1,1,8
3,57.03792,-121.93091,320.94,0.75,0.77,2014-01-01,849,n,249.22,5.81,N,0,4,2014,1,1,8
4,49.92657,-120.63065,329.23,0.67,0.74,2014-01-01,851,n,261.83,5.46,N,0,5,2014,1,1,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2300612,48.90792,-119.29138,326.92,0.56,0.43,2024-12-30,1031,n,268.10,4.92,N,0,2300613,2024,12,365,10
2300613,48.88632,-122.74409,318.54,0.43,0.38,2024-12-30,1031,n,277.93,2.37,N,2,2300614,2024,12,365,10
2300614,48.88623,-122.74597,314.77,0.43,0.38,2024-12-30,1031,n,276.77,1.10,N,0,2300615,2024,12,365,10
2300615,54.52876,-111.87737,367.00,0.51,0.41,2024-12-31,1010,h,264.06,15.90,N,0,2300616,2024,12,366,10


# **(2) Prepare & Merge Fire Detection Data with Land Cover Data**

### Create subset of fire dataset to upload to Google Earth Engine (GEE)

In [ ]:
# Define subset for GEE
fire_coords_2014_2024 = fire_df[['latitude', 'longitude', 'acq_date', 'year']].copy()

# Create index column and rearrange to be the first column
fire_coords_2014_2024['fire_id'] = range(1, len(fire_coords_2014_2024) + 1)
fire_coords_2014_2024 = fire_coords_2014_2024[['fire_id', 'latitude', 'longitude', 'acq_date', 'year']]

# Save as CSV
fire_coords_2014_2024.to_csv('/content/drive/MyDrive/MDP_Capstone_Project/Data/fire_coords_2014_2024.csv', index=False)

# Show sample table
fire_coords_2014_2024

,fire_id,latitude,longitude,acq_date,year
0,1,57.03862,-121.94064,2014-01-01,2014
1,2,56.82743,-121.11518,2014-01-01,2014
2,3,56.82357,-121.11824,2014-01-01,2014
3,4,57.03792,-121.93091,2014-01-01,2014
4,5,49.92657,-120.63065,2014-01-01,2014
...,...,...,...,...,...
2300612,2300613,48.90792,-119.29138,2024-12-30,2024
2300613,2300614,48.88632,-122.74409,2024-12-30,2024
2300614,2300615,48.88623,-122.74597,2024-12-30,2024
2300615,2300616,54.52876,-111.87737,2024-12-31,2024


In [ ]:
# Merge the two dataframes on fire_id
check_df = fire_df.merge(fire_coords_2014_2024, on='fire_id', suffixes=('_orig', '_subset'))

# Check if key columns are the same for all rows
cols_to_check = ['latitude', 'longitude', 'acq_date', 'year']

for col in cols_to_check:
    equal = (check_df[f"{col}_orig"] == check_df[f"{col}_subset"]).all()
    print(f"All values match for {col}: {equal}")

all_match = all(
    (check_df[f"{col}_orig"] == check_df[f"{col}_subset"]).all()
    for col in cols_to_check
)
print("All keys match across fire_id:", all_match)

All values match for latitude: True
All values match for longitude: True
All values match for acq_date: True
All values match for year: True
All keys match across fire_id: True


### Use GEE to Identify Land Cover Information for Each Fire Observation

In [ ]:
!pip install earthengine-api

import ee
ee.Authenticate()   # Log into GEE
ee.Initialize(project='mdp-capstone-project')     # Starts GEE session

# Load uploaded table of fire lat/long points by year (from GEE assets)
fire_points = ee.FeatureCollection("projects/mdp-capstone-project/assets/fire_coords_2014_2024")

# Load MODIS land cover collection
modis = ee.ImageCollection("MODIS/061/MCD12Q1")

# Define the sampling function
def add_land_cover(feature):
    year = ee.Number(feature.get('year'))
    modis_img = modis \
        .filter(ee.Filter.calendarRange(year, year, 'year')) \
        .first() \
        .select('LC_Type1')

    # Sample LC_Type1 at the feature's location
    lc_sample = modis_img.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=feature.geometry(),
        scale=500
    )

    # Get LC_Type1 value from the sampled result
    lc_type1 = lc_sample.get('LC_Type1')

    # Check if LC_Type1 is null
    is_null = ee.Algorithms.IsEqual(lc_type1, None)

    # Assign a default value if null
    lc_type1_value = ee.Algorithms.If(is_null, -1, lc_type1)

    # Add LC_Type1 and null flag as properties to the original features
    return feature.set({'LC_Type1': lc_type1_value})

# Map the sampling function across all fire points
sampled = fire_points.map(add_land_cover)

# Export results to Google Drive as CSV file
task = ee.batch.Export.table.toDrive(
    collection=sampled,
    description='Fire_LandCover_ByYear',
    fileFormat='CSV'
)

# Start the export task
task.start()

### Prepare land cover df for final merge

In [ ]:
import pandas as pd
import json

# Load the GEE-generated fire CSV
fire_LC_df = pd.read_csv('/content/drive/MyDrive/MDP_Capstone_Project/Data/Fire_LandCover_ByYear.csv')

# Identify lat/long values from .geo column
fire_LC_df['latitude'] = fire_LC_df['.geo'].apply(lambda x: json.loads(x)['coordinates'][1])
fire_LC_df['longitude'] = fire_LC_df['.geo'].apply(lambda x: json.loads(x)['coordinates'][0])

# Check result
print(fire_LC_df.head())

           system:index  LC_Type1    acq_date  fire_id  year  \
0  00000000000000000000         8  2014-01-01        1  2014   
1  00000000000000000001        10  2014-01-01        2  2014   
2  00000000000000000002        10  2014-01-01        3  2014   
3  00000000000000000003         8  2014-01-01        4  2014   
4  00000000000000000004         1  2014-01-01        8  2014   

                                                .geo   latitude   longitude  
0  {"type":"Point","coordinates":[-121.9406419713...  57.038618 -121.940642  
1  {"type":"Point","coordinates":[-121.1151821025...  56.827430 -121.115182  
2  {"type":"Point","coordinates":[-121.1182410458...  56.823569 -121.118241  
3  {"type":"Point","coordinates":[-121.9309077537...  57.037922 -121.930908  
4  {"type":"Point","coordinates":[-115.1857682814...  49.291853 -115.185768  


### Merge land cover data with all fire point data

In [ ]:
# Merge on fire ID
merged_df = fire_coords_2014_2024.merge(fire_LC_df, on='fire_id', how='left')

# Check results
print("Rows with LC_Type1:", merged_df['LC_Type1'].notna().sum())
print("Total rows:", len(merged_df))
print("Match %:", merged_df['LC_Type1'].notna().mean() * 100)

merged_df

Rows with LC_Type1: 2300617
Total rows: 2300617
Match %: 100.0


,fire_id,latitude_x,longitude_x,acq_date_x,year_x,system:index,LC_Type1,acq_date_y,year_y,.geo,latitude_y,longitude_y
0,1,57.03862,-121.94064,2014-01-01,2014,00000000000000000000,8,2014-01-01,2014,"{""type"":""Point"",""coordinates"":[-121.9406419713...",57.038618,-121.940642
1,2,56.82743,-121.11518,2014-01-01,2014,00000000000000000001,10,2014-01-01,2014,"{""type"":""Point"",""coordinates"":[-121.1151821025...",56.827430,-121.115182
2,3,56.82357,-121.11824,2014-01-01,2014,00000000000000000002,10,2014-01-01,2014,"{""type"":""Point"",""coordinates"":[-121.1182410458...",56.823569,-121.118241
3,4,57.03792,-121.93091,2014-01-01,2014,00000000000000000003,8,2014-01-01,2014,"{""type"":""Point"",""coordinates"":[-121.9309077537...",57.037922,-121.930908
4,5,49.92657,-120.63065,2014-01-01,2014,00030000000000000000,8,2014-01-01,2014,"{""type"":""Point"",""coordinates"":[-120.6306517229...",49.926570,-120.630652
...,...,...,...,...,...,...,...,...,...,...,...,...
2300612,2300613,48.90792,-119.29138,2024-12-30,2024,0003000000000007b041,8,2024-12-30,2024,"{""type"":""Point"",""coordinates"":[-119.2913830170...",48.907920,-119.291383
2300613,2300614,48.88632,-122.74409,2024-12-30,2024,0003000000000007b042,9,2024-12-30,2024,"{""type"":""Point"",""coordinates"":[-122.7440872582...",48.886320,-122.744087
2300614,2300615,48.88623,-122.74597,2024-12-30,2024,0003000000000007b043,9,2024-12-30,2024,"{""type"":""Point"",""coordinates"":[-122.7459734579...",48.886231,-122.745973
2300615,2300616,54.52876,-111.87737,2024-12-31,2024,000100000000000794f8,4,2024-12-31,2024,"{""type"":""Point"",""coordinates"":[-111.8773739276...",54.528759,-111.877374


In [ ]:
import numpy as np

# Rename columns after merge for clarity
merged_df = merged_df.rename(columns={
    'latitude_x': 'latitude',        # Original fire_df lat
    'longitude_x': 'longitude',      # Original fire_df long
    'acq_date_x': 'acq_date',        # Original fire_df date
    'year_x': 'year',                # Original fire_df year
    'latitude_y': 'latitude_GEE',    # GEE lat
    'longitude_y': 'longitude_GEE',  # GEE long
    'acq_date_y': 'acq_date_GEE',    # GEE date
    'year_y': 'year_GEE'             # GEE year
})

# Check difference between GEE and original coordinates to ensure proper LC mapping
lat_diff = np.sum(~np.isclose(merged_df['latitude'], merged_df['latitude_GEE'], atol=1e-6))
long_diff = np.sum(~np.isclose(merged_df['longitude'], merged_df['longitude_GEE'], atol=1e-6))

# Print results
print(f"Latitude differences outside tolerance: {lat_diff}")
print(f"Longitude differences outside tolerance: {long_diff}")

merged_df.head()

Latitude differences within tolerance: 0
Longitude differences within tolerance: 0


,fire_id,latitude,longitude,acq_date,year,system:index,LC_Type1,acq_date_GEE,year_GEE,.geo,latitude_GEE,longitude_GEE
0,1,57.03862,-121.94064,2014-01-01,2014,00000000000000000000,8,2014-01-01,2014,"{""type"":""Point"",""coordinates"":[-121.9406419713...",57.038618,-121.940642
1,2,56.82743,-121.11518,2014-01-01,2014,00000000000000000001,10,2014-01-01,2014,"{""type"":""Point"",""coordinates"":[-121.1151821025...",56.827430,-121.115182
2,3,56.82357,-121.11824,2014-01-01,2014,00000000000000000002,10,2014-01-01,2014,"{""type"":""Point"",""coordinates"":[-121.1182410458...",56.823569,-121.118241
3,4,57.03792,-121.93091,2014-01-01,2014,00000000000000000003,8,2014-01-01,2014,"{""type"":""Point"",""coordinates"":[-121.9309077537...",57.037922,-121.930908
4,5,49.92657,-120.63065,2014-01-01,2014,00030000000000000000,8,2014-01-01,2014,"{""type"":""Point"",""coordinates"":[-120.6306517229...",49.926570,-120.630652


### Final merge between the original NASA fire dataset with land cover enriched dataset

In [ ]:
# Merge on fire_id
fire_LC_merged = fire_df.merge(
    merged_df[['fire_id', 'LC_Type1']],
    on='fire_id',
    how='left'
)

# Check results
print("Rows with LC_Type1:", fire_LC_merged['LC_Type1'].notna().sum())
print("Total rows:", len(fire_LC_merged))
print("Match %:", fire_LC_merged['LC_Type1'].notna().mean() * 100)

fire_LC_merged

Rows with LC_Type1: 2300617
Total rows: 2300617
Match %: 100.0


,latitude,longitude,brightness,scan,track,acq_date,acq_time,confidence,bright_t31,frp,daynight,type,fire_id,year,month,day_of_year,hour,LC_Type1
0,57.03862,-121.94064,320.09,0.75,0.77,2014-01-01,849,n,248.54,4.45,N,0,1,2014,1,1,8,8
1,56.82743,-121.11518,335.42,0.70,0.75,2014-01-01,849,n,245.92,6.46,N,0,2,2014,1,1,8,10
2,56.82357,-121.11824,328.37,0.70,0.75,2014-01-01,849,n,244.90,6.79,N,0,3,2014,1,1,8,10
3,57.03792,-121.93091,320.94,0.75,0.77,2014-01-01,849,n,249.22,5.81,N,0,4,2014,1,1,8,8
4,49.92657,-120.63065,329.23,0.67,0.74,2014-01-01,851,n,261.83,5.46,N,0,5,2014,1,1,8,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2300612,48.90792,-119.29138,326.92,0.56,0.43,2024-12-30,1031,n,268.10,4.92,N,0,2300613,2024,12,365,10,8
2300613,48.88632,-122.74409,318.54,0.43,0.38,2024-12-30,1031,n,277.93,2.37,N,2,2300614,2024,12,365,10,9
2300614,48.88623,-122.74597,314.77,0.43,0.38,2024-12-30,1031,n,276.77,1.10,N,0,2300615,2024,12,365,10,9
2300615,54.52876,-111.87737,367.00,0.51,0.41,2024-12-31,1010,h,264.06,15.90,N,0,2300616,2024,12,366,10,4


In [ ]:
# Define list of land cover labels
land_cover_labels = {
    -1: "Missing",
    1: "Evergreen Needleleaf Forests",
    2: "Evergreen Broadleaf Forests",
    3: "Deciduous Needleleaf Forests",
    4: "Deciduous Broadleaf Forests",
    5: "Mixed Forests",
    6: "Closed Shrublands",
    7: "Open Shrublands",
    8: "Woody Savannas",
    9: "Savannas",
    10: "Grasslands",
    11: "Permanent Wetlands",
    12: "Croplands",
    13: "Urban and Built-up Lands",
    14: "Cropland/Natural Veg. Mosaic",
    15: "Permanent Snow and Ice",
    16: "Barren",
    17: "Water Bodies",
    255: "Unclassified"
}

# Map labels into new column
fire_LC_merged['LC_Label'] = fire_LC_merged['LC_Type1'].map(land_cover_labels)

# Save merged fire dataset to Google Drive as CSV
fire_df_path = '/content/drive/MyDrive/MDP_Capstone_Project/Data/final_fire_LC_cleaned.csv'
fire_LC_merged.to_csv(fire_df_path, index=False)


# **(3) Complete Last Checks & Adjustments for Merged Fire/LC Dataset**

In [ ]:
# Load df to avoid rerunning previous code chunks
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
fire_LC_merged = pd.read_csv('/content/drive/MyDrive/MDP_Capstone_Project/Data/final_fire_LC_cleaned.csv')

fire_LC_merged

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,latitude,longitude,brightness,scan,track,acq_date,acq_time,confidence,bright_t31,frp,daynight,type,fire_id,year,month,day_of_year,hour,LC_Type1,LC_Label
0,57.03862,-121.94064,320.09,0.75,0.77,2014-01-01,849,n,248.54,4.45,N,0,1,2014,1,1,8,8,Woody Savannas
1,56.82743,-121.11518,335.42,0.70,0.75,2014-01-01,849,n,245.92,6.46,N,0,2,2014,1,1,8,10,Grasslands
2,56.82357,-121.11824,328.37,0.70,0.75,2014-01-01,849,n,244.90,6.79,N,0,3,2014,1,1,8,10,Grasslands
3,57.03792,-121.93091,320.94,0.75,0.77,2014-01-01,849,n,249.22,5.81,N,0,4,2014,1,1,8,8,Woody Savannas
4,49.92657,-120.63065,329.23,0.67,0.74,2014-01-01,851,n,261.83,5.46,N,0,5,2014,1,1,8,8,Woody Savannas
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2300612,48.90792,-119.29138,326.92,0.56,0.43,2024-12-30,1031,n,268.10,4.92,N,0,2300613,2024,12,365,10,8,Woody Savannas
2300613,48.88632,-122.74409,318.54,0.43,0.38,2024-12-30,1031,n,277.93,2.37,N,2,2300614,2024,12,365,10,9,Savannas
2300614,48.88623,-122.74597,314.77,0.43,0.38,2024-12-30,1031,n,276.77,1.10,N,0,2300615,2024,12,365,10,9,Savannas
2300615,54.52876,-111.87737,367.00,0.51,0.41,2024-12-31,1010,h,264.06,15.90,N,0,2300616,2024,12,366,10,4,Deciduous Broadleaf Forests


In [ ]:
# Check for any remaining missing values
missing_summary = fire_LC_merged.isna().sum()
print("Remaining missing values per column:")
print(missing_summary[missing_summary > 0])

# Check all other columns for missing values
print("\nRemaining NaNs in other weather variables:")
print(fire_LC_merged.isna().sum())

Remaining missing values per column:
Series([], dtype: int64)

Remaining NaNs in other weather variables:
latitude       0
longitude      0
brightness     0
scan           0
track          0
acq_date       0
acq_time       0
confidence     0
bright_t31     0
frp            0
daynight       0
type           0
fire_id        0
year           0
month          0
day_of_year    0
hour           0
LC_Type1       0
LC_Label       0
dtype: int64


In [ ]:
# Count how many -1 and 255 values (missing & unclassified) are in LC_Type1
num_missing = (fire_LC_merged['LC_Type1'] == -1).sum()
print(f"Number of missing LC_Type1 values (-1): {num_missing}")

num_missing = (fire_LC_merged['LC_Type1'] == 255).sum()
print(f"Number of unclassified LC_Type1 values (255): {num_missing}")

Number of missing LC_Type1 values (-1): 0
Number of unclassified LC_Type1 values (255): 0


In [ ]:
# Rename columns before saving
fire_LC_merged = fire_LC_merged.rename(columns={
    'latitude': 'Latitude_Fire',
    'longitude': 'Longitude_Fire',
    'brightness': 'Bright_TI4',
    'bright_t31': 'Bright_TI5',
    'frp': 'FRP',
    'scan': 'Scan',
    'track': 'Track',
    'acq_date': 'Acquisition_Date',
    'acq_time': 'Acquisition_Time',
    'confidence': 'Confidence',
    'daynight': 'DayNight',
    'type': 'Fire_Type',
    'fire_id': 'Fire_ID',
    'year': 'Year',
    'month': 'Month',
    'day_of_year': 'Day_of_Year',
    'hour': 'Hour',
})

fire_LC_merged = fire_LC_merged[
    [
        'Fire_ID',
        'Acquisition_Date',
        'Acquisition_Time',
        'Year',
        'Month',
        'Day_of_Year',
        'Hour',
        'Latitude_Fire',
        'Longitude_Fire',
        'Bright_TI4',
        'Bright_TI5',
        'FRP',
        'Scan',
        'Track',
        'Confidence',
        'DayNight',
        'Fire_Type',
        'LC_Type1',
        'LC_Label'
    ]
]

# Save cleaned and land cover-merged fire dataset to Google Drive as CSV
fire_df_path = '/content/drive/MyDrive/MDP_Capstone_Project/Data/Final_Fire_LC_Dataset.csv'
fire_LC_merged.to_csv(fire_df_path, index=False)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

fire_LC_check = pd.read_csv('/content/drive/MyDrive/MDP_Capstone_Project/Data/Final_Fire_LC_Dataset.csv')
fire_LC_check.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Fire_ID,Acquisition_Date,Acquisition_Time,Year,Month,Day_of_Year,Hour,Latitude_Fire,Longitude_Fire,Bright_TI4,Bright_TI5,FRP,Scan,Track,Confidence,DayNight,Fire_Type,LC_Type1,LC_Label
0,1,2014-01-01,849,2014,1,1,8,57.03862,-121.94064,320.09,248.54,4.45,0.75,0.77,n,N,0,8,Woody Savannas
1,2,2014-01-01,849,2014,1,1,8,56.82743,-121.11518,335.42,245.92,6.46,0.70,0.75,n,N,0,10,Grasslands
2,3,2014-01-01,849,2014,1,1,8,56.82357,-121.11824,328.37,244.90,6.79,0.70,0.75,n,N,0,10,Grasslands
3,4,2014-01-01,849,2014,1,1,8,57.03792,-121.93091,320.94,249.22,5.81,0.75,0.77,n,N,0,8,Woody Savannas
4,5,2014-01-01,851,2014,1,1,8,49.92657,-120.63065,329.23,261.83,5.46,0.67,0.74,n,N,0,8,Woody Savannas


In [ ]:
print(fire_LC_check.dtypes)
fire_LC_check.shape

Fire_ID               int64
Acquisition_Date     object
Acquisition_Time      int64
Year                  int64
Month                 int64
Day_of_Year           int64
Hour                  int64
Latitude_Fire       float64
Longitude_Fire      float64
Bright_TI4          float64
Bright_TI5          float64
FRP                 float64
Scan                float64
Track               float64
Confidence           object
DayNight             object
Fire_Type             int64
LC_Type1              int64
LC_Label             object
dtype: object


(2300617, 19)

In [ ]:
# Acquisition_Time format + Hour range checks
acq_str = fire_LC_check['Acquisition_Time'].astype(str).str.strip()

bad_time_format = (~acq_str.str.fullmatch(r"\d{1,4}")).sum()
print("Acquisition_Time bad format rows:", bad_time_format)

bad_hour = (~fire_LC_check['Hour'].between(0, 23)).sum()
print("Hour outside 0–23 rows:", bad_hour)

Acquisition_Time bad format rows: 0
Hour outside 0–23 rows: 0


In [ ]:
print("Exact duplicate rows:", fire_LC_check.duplicated().sum())

# Likely grouping key for “same pixel/time”
subset_key = ['Latitude_Fire','Longitude_Fire','Acquisition_Date','Acquisition_Time']
print("Duplicates on (lat, lon, date, time):",
      fire_LC_check.duplicated(subset=subset_key).sum())

# If you care about daily resolution later:
daily_key = ['Latitude_Fire','Longitude_Fire','Acquisition_Date']
print("Duplicates on (lat, lon, date):",
      fire_LC_check.duplicated(subset=daily_key).sum())

Exact duplicate rows: 0
Duplicates on (lat, lon, date, time): 0
Duplicates on (lat, lon, date): 7


In [ ]:
print("Row count:", len(fire_LC_check))
print("Unique Fire_ID:", fire_LC_check['Fire_ID'].nunique())
print("Fire_ID duplicates:", fire_LC_check['Fire_ID'].duplicated().sum())

Row count: 2300617
Unique Fire_ID: 2300617
Fire_ID duplicates: 0
